# affect_aif Paper Reproduction Notebook

**Open In Colab:** anonymous review repository: <https://anonymous.4open.science/r/affect_aif>. Override `AFFECT_AIF_REPO_URL` if you use a git-cloneable mirror.

This is the mechanical full-paper route. It dry-runs and then runs the numbered paper configs, materializes fresh outputs into the canonical `results/paper/*/raw/` layout, regenerates compact analysis artifacts, and plots the paper readouts.

Use `notebooks/demo.ipynb` first when you want the guided explanation: its default core route is 21 expanded runs, while this full paper route is 1220 expanded runs.

## What This Notebook Proves

This notebook checks the public reproduction contract rather than teaching the whole model. Each experiment section includes:

- the TOML config path;
- a dry-run manifest for run counts;
- the real `scripts/experiment/run.py` execution cell;
- materialization into the canonical `results/paper/` layout;
- analysis and plotting cells for that experiment;
- source-table readouts where the manuscript uses compact tables.

The interpretation still belongs in the manuscript and `docs/results/current.md`. This notebook shows that those paper-facing artifacts can be regenerated from current configs. Full row-level data are also mirrored in the public data packet; see root `README.md`.

## 1. Bootstrap Runtime

In Colab this clones the repo into `/content/affect_aif`. Locally, launch Jupyter from the repo root. GPU is optional; use a GPU runtime if available.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap

IN_COLAB = Path("/content").exists()


def sanitize_repo_url(value):
    value = str(value).strip()
    if value.startswith("[") and "](" in value and value.endswith(")"):
        value = value.split("](", 1)[1][:-1]
    if value.startswith("<") and value.endswith(">"):
        value = value[1:-1]
    return value.strip()


REPO_URL = sanitize_repo_url(os.environ.get("AFFECT_AIF_REPO_URL", "https://anonymous.4open.science/r/affect_aif"))
BRANCH = os.environ.get("AFFECT_AIF_BRANCH", "master")

if IN_COLAB:
    os.chdir("/content")
    if not Path("affect_aif").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "affect_aif"], check=True)
    os.chdir("/content/affect_aif")

ROOT = Path.cwd()
if not (ROOT / "scripts" / "experiment" / "run.py").exists():
    raise RuntimeError("Run this notebook from the affect_aif repo root, or use the Colab bootstrap clone.")

print("Repo root:", ROOT)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

## 2. Install Dependencies

Colab runtimes start clean, so install the project into the runtime. Local users with an existing editable install can set `INSTALL_DEPS = False`.

In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
    print("Installed affect_aif in editable mode for this runtime.")
else:
    print("Skipped dependency install.")

## 3. Check Runtime Device

This reports whether JAX sees an accelerator. The experiments should run on CPU too; the GPU check is for transparency and Colab readiness.

In [ ]:
try:
    import jax
    print("JAX devices:", jax.devices())
except Exception as exc:
    print("JAX device check unavailable:", exc)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU visible. This notebook also runs on CPU; GPU is optional for these trust-task sweeps.")

## 4. Paper Controls And Helpers

`RUN_EXPERIMENTS = True` means every experiment section executes its config before analysis. `MATERIALIZE_RESULTS = True` replaces the canonical raw folder for that experiment with the fresh notebook output so later analysis cannot silently use stale local data.

The dry-run cells show the workload before execution. The complete paper route is 1220 expanded runs; use the demo notebook's 21-run core route first if you only need orientation.

In [ ]:
RUN_EXPERIMENTS = True
MATERIALIZE_RESULTS = True
RUN_ANALYSIS = True
EXPORT_TO_DRIVE = False
DRIVE_PACKET_URL = os.environ.get("AFFECT_AIF_DATA_PACKET_URL")
DRIVE_EXPORT_DEST = Path("/content/drive/MyDrive/affect_aif_results")
RESET_OUTPUTS = False

OUTPUT_ROOT = Path("outputs")
PAPER_BATCH_PREFIX = "paper_full"
MANUSCRIPT_DIR = Path("docs/manuscript")
RESULTS_ROOT = Path("results")

PAPER_CONFIGS = {
    "predictability_value": "configs/paper/01_predictability_value.toml",
    "deployment_ablation": "configs/paper/02_deployment_ablation.toml",
    "partner_selection": "configs/paper/03_partner_selection.toml",
    "betrayal_adaptation": "configs/paper/04_betrayal_adaptation.toml",
    "alpha_sweep": "configs/paper/05a_alpha_sweep.toml",
    "prior_factorial": "configs/paper/05b_prior_factorial.toml",
    "forgiveness": "configs/paper/05c_forgiveness.toml",
}

PAPER_READOUTS = {
    "predictability_value": {
        "section": "Section 3.1",
        "changed": "Predictability and likelihood-evidence readouts are separated from realized payoff.",
        "avoid": "Do not treat binary diagnostic H1 as the main paper readout.",
    },
    "deployment_ablation": {
        "section": "Section 3.2",
        "changed": "Tracked precision only changes behavior when it is deployed into policy precision.",
        "avoid": "Do not describe the result as a broad payoff advantage.",
    },
    "partner_selection": {
        "section": "Section 3.3",
        "changed": "Partner-local precision reaches social allocation in the graded choice regime.",
        "avoid": "Do not turn the result into a one-type preference or payoff headline.",
    },
    "betrayal_adaptation": {
        "section": "Section 3.4",
        "changed": "Accumulated confidence remains behaviorally relevant after an abrupt stance switch.",
        "avoid": "Do not write the result as generic recovery or a clean payoff win.",
    },
    "alpha_sweep": {
        "section": "Section 3.5 / Appendix",
        "changed": "Gain changes confidence amplitude and policy commitment.",
        "avoid": "Do not rank alpha settings by payoff alone.",
    },
    "prior_factorial": {
        "section": "Section 3.5 / Appendix",
        "changed": "Trust priors and precision gain produce distinct calibration profiles.",
        "avoid": "Do not reduce the profile grid to one winning phenotype.",
    },
    "forgiveness": {
        "section": "Section 3.5 / Appendix",
        "changed": "Reengagement and confidence restoration can separate after repair.",
        "avoid": "Do not treat reengagement as automatic confidence recovery.",
    },
}

for name, config in PAPER_CONFIGS.items():
    if not Path(config).exists():
        raise FileNotFoundError(f"Missing paper config for {name}: {config}")

CANONICAL_RAW = {
    "predictability_value": Path("results/paper/01_predictability_value/raw/results.csv"),
    "deployment_ablation": Path("results/paper/02_deployment_ablation/raw/results.csv"),
    "partner_selection": Path("results/paper/03_partner_selection/raw/results.csv"),
    "betrayal_adaptation": Path("results/paper/04_betrayal_adaptation/raw/results.csv"),
    "forgiveness": Path("results/paper/05c_forgiveness/raw/results.csv"),
}

PHENOTYPE_RAW_ROOTS = {
    "alpha_sweep": Path("results/paper/05a_alpha_sweep/raw"),
    "prior_factorial": Path("results/paper/05b_prior_factorial/raw"),
    "forgiveness": Path("results/paper/05c_forgiveness/raw"),
}

PHENOTYPE_CHILD_RAW = {
    "alpha_sweep": [
        PHENOTYPE_RAW_ROOTS["alpha_sweep"] / "open_graded" / "results.csv",
        PHENOTYPE_RAW_ROOTS["alpha_sweep"] / "betrayal" / "results.csv",
    ],
    "prior_factorial": [
        PHENOTYPE_RAW_ROOTS["prior_factorial"] / "open_graded" / "results.csv",
        PHENOTYPE_RAW_ROOTS["prior_factorial"] / "betrayal" / "results.csv",
        PHENOTYPE_RAW_ROOTS["prior_factorial"] / "partner_choice" / "results.csv",
    ],
    "forgiveness": [PHENOTYPE_RAW_ROOTS["forgiveness"] / "results.csv"],
}


SOURCE_TABLES = {
    "deployment_contrast": Path("docs/manuscript/source_tables/h2_deployment_contrast_summary.csv"),
    "deployment_pathway": Path("docs/manuscript/source_tables/h2_deployment_pathway_summary.csv"),
    "partner_choice": Path("docs/manuscript/source_tables/h4_partner_choice_summary.csv"),
    "alpha_sweep": Path("docs/manuscript/source_tables/alpha_sweep/metrics.csv"),
    "prior_factorial": Path("docs/manuscript/source_tables/prior_factorial/metrics.csv"),
    "forgiveness": Path("docs/manuscript/source_tables/forgiveness/metrics.csv"),
}

import pandas as pd
import matplotlib.pyplot as plt


def run_required(cmd, required_path: Path | None = None):
    if required_path is not None and not required_path.exists():
        raise FileNotFoundError(f"Missing required file: {required_path}")
    print("Running:", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)


def batch_name(name: str) -> str:
    return f"{PAPER_BATCH_PREFIX}_{name}"


def batch_dir(name: str) -> Path:
    return OUTPUT_ROOT / batch_name(name)


def dry_run_config(name: str):
    dry_batch = batch_name(name) + "_dry"
    cmd = [
        sys.executable,
        "scripts/experiment/run.py",
        "--config",
        PAPER_CONFIGS[name],
        "--output-dir",
        str(OUTPUT_ROOT),
        "--batch-name",
        dry_batch,
        "--workers",
        "1",
        "--dry-run",
    ]
    run_required(cmd)
    manifest_path = OUTPUT_ROOT / dry_batch / "manifest.json"
    manifest = json.loads(manifest_path.read_text())
    frame = pd.DataFrame(manifest["configs"])
    display(frame[["hypothesis_id", "experiment_id", "rounds", "replications", "expanded_runs", "path"]])
    print("Total expanded runs:", int(frame["expanded_runs"].sum()))
    return frame


def run_paper_config(name: str):
    out = batch_dir(name)
    if RESET_OUTPUTS and out.exists():
        shutil.rmtree(out)
    cmd = [
        sys.executable,
        "scripts/experiment/run.py",
        "--config",
        PAPER_CONFIGS[name],
        "--output-dir",
        str(OUTPUT_ROOT),
        "--batch-name",
        batch_name(name),
        "--workers",
        "1",
    ]
    if RUN_EXPERIMENTS:
        run_required(cmd)
    else:
        print("Experiment run skipped. Set RUN_EXPERIMENTS = True to execute", name)
    if not out.exists():
        raise FileNotFoundError(f"Missing batch output for {name}: {out}")
    return out


def replace_dir(src: Path, dst: Path):
    if not src.exists():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


def materialize_single(name: str, src: Path, dst: Path):
    if MATERIALIZE_RESULTS:
        replace_dir(src, dst)
    if not (dst / "results.csv").exists():
        raise FileNotFoundError(dst / "results.csv")
    return dst / "results.csv"


def materialize_suite(name: str, src_root: Path, dst_root: Path):
    if MATERIALIZE_RESULTS:
        if not src_root.exists():
            raise FileNotFoundError(src_root)
        dst_root.mkdir(parents=True, exist_ok=True)
        child_results = []
        for child in sorted(src_root.iterdir()):
            if not child.is_dir():
                continue
            dst = dst_root / child.name
            replace_dir(child, dst)
            result_path = dst / "results.csv"
            if result_path.exists():
                child_results.append(result_path)
        if not child_results:
            raise RuntimeError(f"No child results available under {src_root}")
    child_results = sorted(path for path in dst_root.glob("*/results.csv"))
    if not child_results:
        raise FileNotFoundError(f"No child results.csv found under {dst_root}")
    return child_results


def analyze_core(name: str):
    raw = CANONICAL_RAW[name]
    if RUN_ANALYSIS:
        run_required(
            [
                sys.executable,
                "scripts/analysis/analyze.py",
                "--results",
                raw,
                "--output-dir",
                raw.parent,
                "--config",
                PAPER_CONFIGS[name],
            ],
            required_path=raw,
        )
    else:
        print("Analysis skipped. Set RUN_ANALYSIS = True to analyze", name)


def analyze_phenotype(name: str):
    raw = PHENOTYPE_RAW_ROOTS[name]
    if RUN_ANALYSIS:
        run_required(
            [
                sys.executable,
                "scripts/analysis/phenotype_artifacts.py",
                name,
                "--results-root",
                raw,
                "--paper-dir",
                MANUSCRIPT_DIR,
            ],
            required_path=raw,
        )
    else:
        print("Analysis skipped. Set RUN_ANALYSIS = True to analyze", name)


def load_csv(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def display_metric_bar(table, title, metric, by="variant_id"):
    if table.empty or metric not in table.columns or by not in table.columns:
        print(f"Cannot plot {title}: missing {metric} or {by}.")
        return
    plot = table.groupby(by, as_index=False)[metric].mean()
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(plot[by].astype(str), plot[metric])
    ax.set(title=title, xlabel=by, ylabel=metric)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.show()



def display_source_table(name: str, columns=None, n=12):
    path = SOURCE_TABLES[name]
    table = load_csv(path)
    print("Source table:", path)
    if columns is not None:
        existing = [col for col in columns if col in table.columns]
        display(table[existing].head(n))
    else:
        display(table.head(n))
    return table

def display_partner_type_mix(table):
    type_col = "selected_partner_type" if "selected_partner_type" in table.columns else "true_type" if "true_type" in table.columns else None
    if type_col is None or "variant_id" not in table.columns:
        display(table.head(12))
        return table
    share_col = next((col for col in ["selection_share", "selected_share", "share", "selection_rate"] if col in table.columns), None)
    if share_col is None:
        display(table.head(12))
        return table
    pivot = table.pivot_table(index=type_col, columns="variant_id", values=share_col, aggfunc="mean").fillna(0.0)
    display((100 * pivot).round(1))
    print("H4 interpretation: allocation is reorganized and entropy is lower; do not treat it as a one-type preference or payoff headline.")
    return pivot

def paper_run_scaffold(name: str, source_path=None):
    readout = PAPER_READOUTS[name]
    print("Reviewer scaffold")
    print("- What changed:", readout["changed"])
    print("- Do not overclaim:", readout["avoid"])
    print("- Paper map:", f"{PAPER_CONFIGS[name]} ({readout['section']})")
    if source_path is not None:
        print("- Evidence source:", source_path)


def best_variant(table, metric, higher=True):
    if table.empty or metric not in table.columns or "variant_id" not in table.columns:
        return "unavailable"
    grouped = table.groupby("variant_id", as_index=False)[metric].mean().dropna()
    if grouped.empty:
        return "unavailable"
    row = grouped.sort_values(metric, ascending=not higher).iloc[0]
    return f"{row.variant_id} ({metric}={row[metric]:.3g})"



def compact_table_readout(label, path, config, changed, avoid, preferred_metrics):
    path = Path(path)
    if not path.exists():
        return {
            "readout": label,
            "source": str(path),
            "config": config,
            "rows": "missing",
            "table_signal": "missing source table; run the section above",
            "do_not_overclaim": avoid,
        }
    table = load_csv(path)
    variant_col = "variant_id" if "variant_id" in table.columns else None
    signals = []
    for metric, higher in preferred_metrics:
        if metric in table.columns and variant_col is not None:
            signals.append(best_variant(table, metric, higher=higher))
        elif metric in table.columns:
            value = pd.to_numeric(table[metric], errors="coerce").mean()
            if pd.notna(value):
                signals.append(f"mean {metric}={value:.3g}")
    if not signals:
        numeric_cols = list(table.select_dtypes(include="number").columns[:4])
        signals.append("numeric columns: " + ", ".join(numeric_cols) if numeric_cols else "inspect table columns")
    return {
        "readout": label,
        "source": str(path),
        "config": config,
        "rows": len(table),
        "table_signal": "; ".join(signals),
        "what_changed": changed,
        "do_not_overclaim": avoid,
    }

## 5. Predictability-Value: Run And Analyze

Config: `configs/paper/01_predictability_value.toml`

This section reproduces the predictability-over-value readout used in Section 3.1. It materializes fresh output under `results/paper/01_predictability_value/raw/`.

In [ ]:
pv_manifest = dry_run_config("predictability_value")
pv_batch = run_paper_config("predictability_value")
pv_raw = materialize_single(
    "predictability_value",
    pv_batch / "predictability_value" / "predictability_value",
    CANONICAL_RAW["predictability_value"].parent,
)
analyze_core("predictability_value")
print("Predictability-value raw result:", pv_raw)
paper_run_scaffold("predictability_value", pv_raw)

## 6. Deployment Ablation: Run And Analyze

Config: `configs/paper/02_deployment_ablation.toml`

This section reproduces the tracked-only deployment ablation used in Section 3.2. The current paper-scale readout is lower entropy from beta-to-gamma deployment with nearly matched payoff, not a broad payoff advantage.

In [ ]:
deployment_manifest = dry_run_config("deployment_ablation")
deployment_batch = run_paper_config("deployment_ablation")
deployment_raw = materialize_single(
    "deployment_ablation",
    deployment_batch / "deployment_ablation" / "deployment_ablation",
    CANONICAL_RAW["deployment_ablation"].parent,
)
analyze_core("deployment_ablation")
deployment_contrast = display_source_table("deployment_contrast")
deployment_pathway = display_source_table("deployment_pathway")
print("Deployment-ablation raw result:", deployment_raw)
print("H2 readout: lower entropy through deployment, with nearly matched payoff.")
paper_run_scaffold("deployment_ablation", SOURCE_TABLES["deployment_pathway"])

## 7. Partner Selection: Run And Analyze

Config: `configs/paper/03_partner_selection.toml`

This section reproduces the graded partner-selection readout used in Section 3.3. The 30-seed result supports allocation reorganization and lower entropy; it does not support a simple cooperator-preference or payoff headline.

In [ ]:
selection_manifest = dry_run_config("partner_selection")
selection_batch = run_paper_config("partner_selection")
selection_raw = materialize_single(
    "partner_selection",
    selection_batch / "partner_selection" / "partner_selection",
    CANONICAL_RAW["partner_selection"].parent,
)
analyze_core("partner_selection")
partner_choice_table = display_source_table("partner_choice")
display_partner_type_mix(partner_choice_table)
print("Partner-selection raw result:", selection_raw)
paper_run_scaffold("partner_selection", SOURCE_TABLES["partner_choice"])

## 8. Betrayal Adaptation: Run And Analyze

Config: `configs/paper/04_betrayal_adaptation.toml`

This is the paper betrayal-adaptation confirmation run. It materializes fresh output under `results/paper/04_betrayal_adaptation/raw/`.

In [ ]:
h5_manifest = dry_run_config("betrayal_adaptation")
h5_batch = run_paper_config("betrayal_adaptation")
h5_raw = materialize_single(
    "betrayal_adaptation",
    h5_batch / "betrayal_adaptation" / "betrayal_adaptation",
    CANONICAL_RAW["betrayal_adaptation"].parent,
)
analyze_core("betrayal_adaptation")
print("Betrayal-adaptation raw result:", h5_raw)

### H5 Betrayal-Adaptation Readout

This plots a generated H5 summary table when available and shows the first rows for inspection.

In [ ]:
h5_phase_path = CANONICAL_RAW["betrayal_adaptation"].parent / "analysis" / "raw" / "betrayal_phase_summary.csv"
h5_final_path = CANONICAL_RAW["betrayal_adaptation"].parent / "analysis" / "raw" / "final_round_summary.csv"
h5_table = load_csv(h5_phase_path) if h5_phase_path.exists() else load_csv(h5_final_path)
display(h5_table.head())
for candidate in ["q_pi_entropy", "joint_accuracy", "total_payoff", "payoff"]:
    if candidate in h5_table.columns:
        display_metric_bar(h5_table, f"H5 betrayal adaptation: {candidate}", candidate)
print("Betrayal readout: compare entropy, accuracy, and payoff artifacts before making stronger behavioral claims.")
paper_run_scaffold("betrayal_adaptation", h5_phase_path if h5_phase_path.exists() else h5_final_path)

## 9. Exp A Alpha Sweep: Run And Analyze

Config: `configs/paper/05a_alpha_sweep.toml`

This config expands into two paper sub-experiments, `open_graded` and `betrayal`. The section runs both, materializes `results/paper/05a_alpha_sweep/raw/open_graded/results.csv` and `results/paper/05a_alpha_sweep/raw/betrayal/results.csv`, and regenerates the compact phenotype metrics and figure.

In [ ]:
alpha_manifest = dry_run_config("alpha_sweep")
alpha_batch = run_paper_config("alpha_sweep")
alpha_raw = materialize_suite("alpha_sweep", alpha_batch / "exp_a", PHENOTYPE_RAW_ROOTS["alpha_sweep"])
analyze_phenotype("alpha_sweep")
print("Exp A raw result:", alpha_raw)

### Exp A Alpha-Sweep Readout

This plots late entropy from the regenerated compact `metrics.csv`.

In [ ]:
alpha_metrics = load_csv("results/paper/05a_alpha_sweep/raw/metrics.csv")
display(alpha_metrics.head())
alpha_source = display_source_table("alpha_sweep")
display_metric_bar(alpha_metrics, "Exp A: alpha sweep late entropy", "entropy_late")
print("Exp A readout: lowest late entropy:", best_variant(alpha_metrics, "entropy_late", higher=False))
paper_run_scaffold("alpha_sweep", SOURCE_TABLES["alpha_sweep"])

## 10. Exp B Prior Factorial: Run And Analyze

Config: `configs/paper/05b_prior_factorial.toml`

This config expands into `open_graded`, `betrayal`, and `partner_choice`. The section materializes all children under `results/paper/05b_prior_factorial/raw/` and regenerates compact phenotype artifacts.

In [ ]:
prior_manifest = dry_run_config("prior_factorial")
prior_batch = run_paper_config("prior_factorial")
prior_raw = materialize_suite("prior_factorial", prior_batch / "exp_b", PHENOTYPE_RAW_ROOTS["prior_factorial"])
analyze_phenotype("prior_factorial")
print("Exp B raw result:", prior_raw)

### Exp B Prior-Factorial Readout

This plots trust asymmetry from the regenerated compact `metrics.csv`.

In [ ]:
prior_metrics = load_csv("results/paper/05b_prior_factorial/raw/metrics.csv")
display(prior_metrics.head())
prior_source = display_source_table("prior_factorial")
display_metric_bar(prior_metrics, "Exp B: prior factorial trust asymmetry", "trust_asymmetry")
print("Exp B readout: strongest trust asymmetry:", best_variant(prior_metrics, "trust_asymmetry", higher=True))
paper_run_scaffold("prior_factorial", SOURCE_TABLES["prior_factorial"])

## 11. Exp C Forgiveness: Run And Analyze

Config: `configs/paper/05c_forgiveness.toml`

This section runs the trust-repair/reengagement experiment, materializes the raw folder, and regenerates the forgiveness metrics and figure.

In [ ]:
forgiveness_manifest = dry_run_config("forgiveness")
forgiveness_batch = run_paper_config("forgiveness")
forgiveness_raw = materialize_single(
    "forgiveness",
    forgiveness_batch / "exp_c" / "forgiveness",
    PHENOTYPE_RAW_ROOTS["forgiveness"],
)
analyze_phenotype("forgiveness")
print("Exp C raw result:", forgiveness_raw)

### Exp C Forgiveness Readout

This plots reengagement from the regenerated compact `metrics.csv`.

In [ ]:
forgiveness_metrics = load_csv("results/paper/05c_forgiveness/raw/metrics.csv")
display(forgiveness_metrics.head())
forgiveness_source = display_source_table("forgiveness")
display_metric_bar(forgiveness_metrics, "Exp C: reengagement rate", "reengagement_rate")
print("Exp C readout: strongest reengagement:", best_variant(forgiveness_metrics, "reengagement_rate", higher=True))
paper_run_scaffold("forgiveness", SOURCE_TABLES["forgiveness"])

## 12. Verify Result Packet Cleanliness

The local `results/` folder should be safe to copy to Drive: no checkpoints, no partial CSVs, raw files under semantic paper folders, and compact summaries beside each result folder. Row-level paper data should be accessible either from this generated folder or from the public data packet in root `README.md`.

In [ ]:
partial_files = sorted(Path("results").glob("**/results_partial.csv")) + sorted(Path("results").glob("**/checkpoint_manifest.json"))
if partial_files:
    raise RuntimeError("Partial/checkpoint files remain:\n" + "\n".join(map(str, partial_files[:20])))

raw_count = sum(1 for _ in Path("results/paper").glob("**/raw/**/results.csv")) if Path("results/paper").exists() else 0
print("Paper raw results.csv count:", raw_count)

for name, path in CANONICAL_RAW.items():
    print(f"{name:22s}", "OK" if path.exists() else "missing", path)
for name, paths in PHENOTYPE_CHILD_RAW.items():
    status = "OK" if all(path.exists() for path in paths) else "missing"
    print(f"{name:22s}", status, ", ".join(map(str, paths)))

if shutil.which("git"):
    probe = Path("results/paper/05a_alpha_sweep/raw/open_graded/results.csv")
    if probe.exists():
        check = subprocess.run(["git", "check-ignore", "-q", str(probe)], check=False)
        print("Raw result gitignored:", check.returncode == 0)

## 13. Final Interpretation Scaffold

This cell builds a compact readout from the regenerated source tables and metrics. It is a reproducibility scaffold: it points to the table evidence, names the intended interpretation, and keeps the common overclaim beside each result.

In [ ]:
final_scaffold_specs = [
    {
        "label": "Section 3.2 deployment pathway",
        "path": SOURCE_TABLES["deployment_pathway"],
        "config": PAPER_CONFIGS["deployment_ablation"],
        "changed": "Beta-to-gamma deployment should lower policy entropy relative to tracked-only.",
        "avoid": "Do not describe the result as a broad payoff advantage.",
        "metrics": [("mean_q_pi_entropy", False), ("total_payoff", True)],
    },
    {
        "label": "Section 3.3 partner selection",
        "path": SOURCE_TABLES["partner_choice"],
        "config": PAPER_CONFIGS["partner_selection"],
        "changed": "Partner-local precision reorganizes graded partner allocation.",
        "avoid": "Do not turn the result into a one-type preference or payoff headline.",
        "metrics": [("selection_rate", True), ("mean_q_pi_entropy", False), ("mean_payoff", True)],
    },
    {
        "label": "Section 3.5 alpha sweep",
        "path": SOURCE_TABLES["alpha_sweep"],
        "config": PAPER_CONFIGS["alpha_sweep"],
        "changed": "Gain changes confidence amplitude and policy commitment.",
        "avoid": "Do not rank alpha settings by payoff alone.",
        "metrics": [("entropy_late", False), ("beta_range", True), ("mean_payoff", True)],
    },
    {
        "label": "Section 3.5 prior x gain profiles",
        "path": SOURCE_TABLES["prior_factorial"],
        "config": PAPER_CONFIGS["prior_factorial"],
        "changed": "Starting beliefs and gain produce distinct calibration profiles.",
        "avoid": "Do not reduce the profile grid to one winning phenotype.",
        "metrics": [("trust_asymmetry", True), ("entropy_late", False), ("mean_payoff", True)],
    },
    {
        "label": "Section 3.5 forgiveness / trust repair",
        "path": SOURCE_TABLES["forgiveness"],
        "config": PAPER_CONFIGS["forgiveness"],
        "changed": "Reengagement and confidence restoration can separate after repair.",
        "avoid": "Do not treat reengagement as automatic confidence recovery.",
        "metrics": [("reengagement_rate", True), ("payoff_recovery", True), ("beta_recovery_r200", True)],
    },
]

final_scaffold = pd.DataFrame(
    compact_table_readout(
        spec["label"],
        spec["path"],
        spec["config"],
        spec["changed"],
        spec["avoid"],
        spec["metrics"],
    )
    for spec in final_scaffold_specs
)
display(final_scaffold)
print("Use this as a table-derived orientation aid. The manuscript and docs/results/current.md remain the interpretation source of truth.")